In [1]:
!wget -q -O darija_parallel.jsonl \
  https://media.githubusercontent.com/media/darija-open-dataset/dataset/main/doda-500k/darija_parallel.jsonl
!ls -lh darija_parallel.jsonl

-rw-r--r-- 1 root root 219M Oct 20 21:49 darija_parallel.jsonl


In [2]:
import pandas as pd

PATH = "darija_parallel.jsonl"

df = pd.read_json(PATH, lines=True)

In [3]:
df.columns

Index(['id', 'darija_arabic', 'darija_latin', 'english', 'code_switch',
       'register', 'domain', 'generation_mode', 'source'],
      dtype='object')

In [4]:
df["register"].unique()

array(['neutral', 'casual', 'very_casual', 'polite', None], dtype=object)

In [5]:
df["generation_mode"].unique()

array(['doda_grounded', 'exploratory', 'doda_human'], dtype=object)

In [6]:
for mode in ["doda_grounded","exploratory"]:
  print(df["source"][df["generation_mode"]==mode].unique())

['gemini-3.1-flash-lite']
['gemini-3.1-flash-lite']


In [7]:
import re

def norm(s):
    s = re.sub(r"[^\w\s]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

hum = df[df["generation_mode"] == "doda_human"].copy()
hum["eng_n"] = hum["english"].map(norm)

print("Lignes humaines      :", len(hum))
print("Anglais vide         :", (hum["eng_n"] == "").sum())
print("Darija arabe vide    :", hum["darija_arabic"].isna().sum())
print("Paires identiques    :", hum.duplicated(["eng_n", "darija_arabic"]).sum())
print("Phrases anglaises uniques :", hum["eng_n"].nunique())

Lignes humaines      : 48350
Anglais vide         : 0
Darija arabe vide    : 0
Paires identiques    : 372
Phrases anglaises uniques : 44652


In [8]:
def ratio_arabe(t):
    lettres = [c for c in str(t) if c.isalpha()]
    if not lettres:
        return 0.0
    return sum("\u0600" <= c <= "\u06FF" for c in lettres) / len(lettres)

hum["r_ar"] = hum["darija_arabic"].map(ratio_arabe)
print("Darija < 70 % arabe :", (hum["r_ar"] < 0.7).sum())

n_trad = hum.groupby("eng_n")["darija_arabic"].nunique()
print("Phrases avec >1 traduction :", (n_trad > 1).sum())
print(n_trad.sort_values(ascending=False).head(5))

Darija < 70 % arabe : 0
Phrases avec >1 traduction : 2175
eng_n
leave me alone         23
god help you           21
where are you going    17
mind your business     16
hurry up               15
Name: darija_arabic, dtype: int64


Première version : ratio sur tous les caractères → 31 lignes rejetées, toutes valides (chiffres, ponctuation). Correction : ratio sur les lettres seulement → 0 ligne.

In [9]:
print(hum.loc[hum["r_ar"] < 0.7, ["english", "darija_arabic"]].to_string())

Empty DataFrame
Columns: [english, darija_arabic]
Index: []


In [10]:
df["generation_mode"].value_counts()

,count
generation_mode,
doda_grounded,226000
exploratory,226000
doda_human,48350
